In [1]:
import pandas as pd

Load Dataset Simple analysis

In [11]:
from pathlib import Path
import pandas as pd

folder = Path("D:/Projects/chicago-bike-demand-intelligence/01_data_raw/divvy")

files = list(folder.glob("*.csv"))

print(f"Found {len(files)} files")

df_list = []

for file in files:
    print(f"Reading: {file.name}")
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

print(df.head())
print(df.shape)

Found 3 files
Reading: 202503-divvy-tripdata.csv
Reading: 202504-divvy-tripdata.csv
Reading: 202505-divvy-tripdata.csv
            ride_id  rideable_type               started_at  \
0  16CBE9844D401954  electric_bike  2025-03-18 08:39:20.065   
1  1CB408029E2B5F74  electric_bike  2025-03-24 16:04:22.239   
2  7B6A76CD0F204D08  electric_bike  2025-03-10 16:06:19.708   
3  4F7084E3D75CDE31  electric_bike  2025-03-21 14:28:14.579   
4  E419A570A5A0475B  electric_bike  2025-03-14 17:54:14.484   

                  ended_at start_station_name start_station_id  \
0  2025-03-18 08:51:37.633                NaN              NaN   
1  2025-03-24 16:27:41.347                NaN              NaN   
2  2025-03-10 16:29:17.457                NaN              NaN   
3  2025-03-21 14:35:06.160                NaN              NaN   
4  2025-03-14 18:17:53.254                NaN              NaN   

                end_station_name end_station_id  start_lat  start_lng  \
0        Canal St & Jackson Blvd

Convert timestamps

In [12]:
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

Create Trip Duration

In [13]:
df['trip_duration_min'] = (
    df['ended_at'] - df['started_at']
).dt.total_seconds() / 60

Extract time features

In [14]:
df['date'] = df['started_at'].dt.date
df['hour'] = df['started_at'].dt.hour
df['day_of_week'] = df['started_at'].dt.dayofweek
df['month'] = df['started_at'].dt.month

In [ ]:
Add simple flags

In [15]:
df['is_weekend'] = df['day_of_week'].isin([5, 6])
df['rush_hour'] = df['hour'].isin([7,8,9,16,17,18])

Basic cleaning

In [16]:
df = df[df['trip_duration_min'] > 0]
df = df[df['trip_duration_min'] < 180]  # remove extreme outliers

In [17]:
print(df.head())
print(df[['trip_duration_min']].describe())
print(df.isna().sum())

            ride_id  rideable_type              started_at  \
0  16CBE9844D401954  electric_bike 2025-03-18 08:39:20.065   
1  1CB408029E2B5F74  electric_bike 2025-03-24 16:04:22.239   
2  7B6A76CD0F204D08  electric_bike 2025-03-10 16:06:19.708   
3  4F7084E3D75CDE31  electric_bike 2025-03-21 14:28:14.579   
4  E419A570A5A0475B  electric_bike 2025-03-14 17:54:14.484   

                 ended_at start_station_name start_station_id  \
0 2025-03-18 08:51:37.633                NaN              NaN   
1 2025-03-24 16:27:41.347                NaN              NaN   
2 2025-03-10 16:29:17.457                NaN              NaN   
3 2025-03-21 14:35:06.160                NaN              NaN   
4 2025-03-14 18:17:53.254                NaN              NaN   

                end_station_name end_station_id  start_lat  start_lng  \
0        Canal St & Jackson Blvd          13138      41.91     -87.67   
1  Albany Ave & Bloomingdale Ave          15655      41.86     -87.68   
2  Albany Ave & B

In [18]:
output_path = Path("D:/Projects/chicago-bike-demand-intelligence/02_data_processed/cleaned_divvy_trips.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print("Cleaned data saved")
print(output_path)

Cleaned data saved
D:\Projects\chicago-bike-demand-intelligence\02_data_processed\cleaned_divvy_trips.csv


In [19]:
print(df[["trip_duration_min"]].describe())
print(df.isna().sum())

       trip_duration_min
count       1.168919e+06
mean        1.299067e+01
std         1.426396e+01
min         1.100000e-03
25%         5.078850e+00
50%         8.867350e+00
75%         1.559630e+01
max         1.799700e+02
ride_id                    0
rideable_type              0
started_at                 0
ended_at                   0
start_station_name    229683
start_station_id      229683
end_station_name      237683
end_station_id        237683
start_lat                  0
start_lng                  0
end_lat                    7
end_lng                    7
member_casual              0
trip_duration_min          0
date                       0
hour                       0
day_of_week                0
month                      0
is_weekend                 0
rush_hour                  0
dtype: int64
